In [ ]:
# rende config.py (in notebooks/) importabile anche dai notebook in sottocartelle
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import matplotlib.pyplot as plt
import concurrent.futures
from concurrent.futures import ProcessPoolExecutor
import logging
import albedo_functions as af


In [ ]:
# Configurazione logging
log_file = "04-Fig4_BIAS_plot_seasons.log"
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s",
                    handlers=[logging.FileHandler(log_file)])


In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
var = 'tas'
seasons = ['DJF', 'MAM', 'JJA', 'SON']
SAVE_PATH = f'{WORK_DIR}/'


In [ ]:
def plot_bias_season(exp_ctrl, exp_sens, var, y1, y2, season, save_path):
    """Mappe di bias (control / sensitivity) per un range di lead year e una stagione."""
    lead = f"{y1}-{y2}"
    try:
        dset_ctrl = xr.open_mfdataset(f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_bias_{season}.nc')
        dset_sens = xr.open_mfdataset(f'{save_path}/{exp_sens}_{var}_lead_{lead}_bias_{season}.nc')
        dset_ctrl_p = xr.open_mfdataset(f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_bias_p_{season}.nc')
        dset_sens_p = xr.open_mfdataset(f'{save_path}/{exp_sens}_{var}_lead_{lead}_bias_p_{season}.nc')

        levels = [-0.5, -0.4, -0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3, 0.4, 0.5]
        af.map_plot(dset_ctrl[var], dset_ctrl_p["p"], levels=levels,
                    title=f'a) DCPP-CTRL {season} {lead}', cmap='BrBG', antartica=False)
        plt.savefig(f'{FIG_DIR}/{exp_ctrl}_{var}_bias_{season}_{lead}.png', dpi=600, bbox_inches='tight')
        plt.close('all')

        af.map_plot(dset_sens[var], dset_sens_p["p"], levels=levels,
                    title=f'b) DCPP-SENS {season} {lead}', cmap='BrBG', antartica=False)
        plt.savefig(f'{FIG_DIR}/{exp_sens}_{var}_bias_{season}_{lead}.png', dpi=600, bbox_inches='tight')
        plt.close('all')
        logging.info(f"mappe bias salvate {season} {lead}")
    except Exception as e:
        logging.exception(f"Error {season} {lead}: {e}")


In [ ]:
%%time
with concurrent.futures.ProcessPoolExecutor(max_workers=2) as executor:
    futures = {}
    for season in seasons:
        for y1 in range(5):
            for y2 in range(y1 + 1, 5):
                fut = executor.submit(plot_bias_season, exp_ctrl, exp_sens, var, y1, y2, season, SAVE_PATH)
                futures[fut] = f"{season} {y1}-{y2}"
    for _i, _f in enumerate(concurrent.futures.as_completed(futures), 1):
        _f.result()
        print(f"  [{futures[_f]}] completato {_i}/{len(futures)}", flush=True)
